In [1]:
from pathlib import Path 
import sys 

import numpy as np 
import pandas as pd 

PROJECT_ROOT = Path.cwd() 

if PROJECT_ROOT.name == "notebooks": 
    PROJECT_ROOT = PROJECT_ROOT.parent 

sys.path.insert(0, str(PROJECT_ROOT))

from src.vector_store import VectorStore 

embeddings = np.load(PROJECT_ROOT / "data/embeddings/abo_image_embeddings_4500.npy")

metadata = pd.read_csv(PROJECT_ROOT / "data/embeddings/abo_image_metadata_4500.csv") 

print(embeddings.shape) 
print(len(metadata))

(4500, 512)
4500


In [2]:
collection_name = "abo_products_4500" 

In [3]:
store_path = PROJECT_ROOT / "qdrant_storage_4500" 

vector_store = VectorStore(
    storage_path=str(store_path), 
    collection_name=collection_name, 
    vector_size=embeddings.shape[1], 
)

In [4]:
if vector_store.client.collection_exists(collection_name): 
    vector_store.client.delete_collection(collection_name) 

vector_store.create_collection() 
print("Collection created.") 

Collection created.


In [5]:
vector_store.index_embeddings(
    embeddings=embeddings, 
    metadata=metadata, 
)

print("Stored points:", vector_store.count()) 

Stored points: 4500


In [6]:
vector_store.close() 

reloaded_store = VectorStore(
    storage_path=str(store_path),
    collection_name=collection_name,
    vector_size=embeddings.shape[1],
)

print("Reloaded count:", reloaded_store.count())

Reloaded count: 4500


In [8]:
# Testing resuable store 

from src.embedder import Embedder 

embedder = Embedder() 
query_embedding = embedder.embed_text("a black handbag") 

results = reloaded_store.search(
    query_embedding=query_embedding, 
    limit=5, 
)

for result in results: 
    print(result) 

/Users/aob/AOB/CS/Projects/clip-semantic-search/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1109 10:46:51.578000 88494 .venv/lib/python3.13/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 75239.24it/s]


{'id': 4308, 'score': 0.3314566491195498, 'item_id': 'B0767VBGDK', 'image_id': '81S8x-UMEfL', 'product_type': 'HANDBAG', 'title': 'The Fix Tiffni Geometric Hard Shell Chain Strap Crossbody Bag, black/metallic', 'image_path': 'data/ABO/abo-images-small/images/small/c2/c222622e.jpg'}
{'id': 4196, 'score': 0.3271665647401949, 'item_id': 'B07WRJG7RM', 'image_id': '8165RQFoRmL', 'product_type': 'HANDBAG', 'title': "Flavia Women's Handbag (Black)", 'image_path': 'data/ABO/abo-images-small/images/small/48/486ab0b7.jpg'}
{'id': 4246, 'score': 0.3249581046972327, 'item_id': 'B07XFWWZKJ', 'image_id': '81ExuT2SBML', 'product_type': 'HANDBAG', 'title': "Nia & Nicole Women's Handbag (Black)", 'image_path': 'data/ABO/abo-images-small/images/small/d6/d63e0cac.jpg'}
{'id': 4241, 'score': 0.3225569355422033, 'item_id': 'B074CSRSPK', 'image_id': '81E2YJOMuQL', 'product_type': 'HANDBAG', 'title': 'Amazon Brand - The Fix Zoe Studded Metallic Bowler Satchel', 'image_path': 'data/ABO/abo-images-small/images

In [9]:
assert len(results) == 5
assert results[0]["score"] >= results[-1]["score"]
assert "image_path" in results[0]
assert "product_type" in results[0]

In [10]:
reloaded_store.client.close()

In [11]:
vector_store.client.close()